
<div style="background-color:#1F3864; padding:16px; border-radius:6px;">
<h1 style="color:#FFFFFF; font-family:Arial; margin:0;">Day 204 &mdash; MCP Server: Wrapping ResumeGapAI as a Callable Tool</h1>
<p style="color:#D9E8F5; font-family:Arial; margin:6px 0 0 0;">Month 13, Week 2, Day 1 &mdash; Agentic AI &amp; Advanced GenAI Portfolio</p>
</div>

**Scenario:** ResumeGapAI's locked TF-IDF engine (`resumegap_engine.py`, deployed at resumegapai.streamlit.app)
currently only has one caller &mdash; the Streamlit UI. Today it gets a second caller: an **MCP server**, so any
MCP-compatible LLM client (Claude Desktop, an LLM agent, LangGraph via an MCP client node) can call
`analyze_resume_gap()` as a tool, without knowing anything about scikit-learn or TF-IDF internally.

**Scope today:** one MCP server, one tool, wrapping one already-working function. No new engine logic,
no LLM calls inside the server, no multi-tool servers yet (that's the Week 2 mini-project). The entire lesson
is about the *boundary* between your Python function and the protocol that exposes it &mdash; input validation,
error handling, and testing without a live client.

**Stack (verified by installing and running, not assumed):** `fastmcp==3.4.4`, `pydantic==2.13.4`,
`scikit-learn==1.8.0` (Colab defaults at time of writing &mdash; pin these explicitly, they move fast).



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 1 &mdash; Raw Data (locked, do not modify)</h2>
</div>

`resumegap_engine.py` below is byte-for-byte the Day 199 locked, deployed version. It is the Raw Data layer
for today &mdash; all of today's work happens in a *new* file, `resumegap_mcp_server.py`, that imports from it.


In [1]:

# Install pinned versions (Colab default sklearn/pydantic drift over time — always verify, never assume)
!pip install fastmcp==3.4.4 pydantic==2.13.4 --quiet


In [2]:

%%writefile resumegap_engine.py
"""
resumegap_engine.py
====================
LOCKED — Day 199 finalized core engine for ResumeGapAI (deployed at
resumegapai.streamlit.app). Raw Data layer for Day 204. Do NOT modify.
"""

import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

BACKGROUND_CORPUS = [
    "experienced professional seeking growth opportunities in a fast paced environment",
    "responsible for managing projects and delivering results on time and within budget",
    "strong communication and collaboration skills working with cross functional teams",
    "proficient in Microsoft Office including Excel Word and PowerPoint for reporting",
    "bachelor degree in relevant field with several years of professional experience",
    "seeking a motivated candidate with strong analytical and problem solving skills",
    "must have excellent written and verbal communication skills and attention to detail",
    "experience working in an agile environment with sprint planning and stand ups",
    "candidate should have strong leadership skills and experience managing a team",
    "looking for someone who can work independently and also collaborate with stakeholders",
]


def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tfidf_similarity_naive(resume: str, jd: str) -> float:
    docs = [clean_text(resume), clean_text(jd)]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    sim = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]
    return round(sim * 100, 2)


def tfidf_similarity_fixed(resume: str, jd: str) -> float:
    docs = BACKGROUND_CORPUS + [clean_text(resume), clean_text(jd)]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    resume_vec = matrix[-2:-1]
    jd_vec = matrix[-1:]
    sim = cosine_similarity(resume_vec, jd_vec)[0][0]
    return round(sim * 100, 2)


def gap_words(resume: str, jd: str, top_n: int = 8) -> list:
    resume_clean = clean_text(resume)
    jd_clean = clean_text(jd)
    docs = BACKGROUND_CORPUS + [resume_clean, jd_clean]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    feature_names = vec.get_feature_names_out()

    jd_vector = matrix[-1].toarray()[0]
    resume_words = set(resume_clean.split())

    scored = [
        (feature_names[i], jd_vector[i])
        for i in range(len(feature_names))
        if jd_vector[i] > 0 and feature_names[i] not in resume_words
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [word for word, _score in scored[:top_n]]


def match_band(score: float) -> str:
    if score >= 70:
        return "Strong Match"
    elif score >= 45:
        return "Moderate Match"
    else:
        return "Weak Match"


def analyze_resume_gap(resume: str, jd: str, top_n: int = 8) -> dict:
    score = tfidf_similarity_fixed(resume, jd)
    band = match_band(score)
    gaps = gap_words(resume, jd, top_n)
    return {"similarity_score": score, "match_band": band, "gap_words": gaps}


Writing resumegap_engine.py


In [3]:

# Three test resume/JD pairs for today (P1-P3). Not random-seeded like tabular days —
# fixed realistic text pairs, same convention as Day198/199.
pairs = {
    "P1_DataAnalyst": {
        "resume": "Data Analyst with 3 years experience in SQL, Excel, and Power BI dashboards. "
                  "Built weekly sales reports and cleaned raw data using pandas. Comfortable "
                  "presenting insights to stakeholders.",
        "jd": "Seeking a Data Analyst skilled in SQL and Python, with hands-on experience in AWS "
              "and Tableau for cloud-based reporting. Must communicate findings clearly to "
              "non-technical teams."
    },
    "P2_MLEngineer": {
        "resume": "Machine Learning Engineer experienced in Python, scikit-learn, and deploying "
                  "models with Flask. Built classification pipelines and monitored model drift "
                  "in production.",
        "jd": "Looking for a Machine Learning Engineer with strong PyTorch and Docker skills, "
              "plus experience running models on Kubernetes and AWS. MLOps and CI/CD experience "
              "required."
    },
    "P3_ProductManager": {
        "resume": "Product Manager who led roadmap planning and worked closely with engineering "
                  "and design teams. Ran user interviews and prioritized features using RICE "
                  "scoring.",
        "jd": "Seeking a Product Manager with strong SQL skills for data-driven decisions, "
              "experience running A/B tests, and familiarity with Agile and Jira for sprint "
              "planning."
    },
}
for k, v in pairs.items():
    print(k, "-> resume", len(v["resume"]), "chars, jd", len(v["jd"]), "chars")


P1_DataAnalyst -> resume 187 chars, jd 178 chars
P2_MLEngineer -> resume 167 chars, jd 169 chars
P3_ProductManager -> resume 159 chars, jd 162 chars



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 2 &mdash; Concept Notes</h2>
</div>

### 2.1 What MCP actually is
The Model Context Protocol is a JSON-RPC-based spec for exposing three primitives to an LLM client:
**tools** (callable functions with side effects or computation &mdash; what we're building today),
**resources** (read-only data, like GET endpoints), and **prompts** (reusable templates). A server you
write once is consumable by any MCP-compatible client (Claude Desktop, LangGraph via an MCP client node,
Cursor, etc.) &mdash; you're not writing a bespoke integration per client.

### 2.2 FastMCP: server, tool, client
`FastMCP` (the standalone `fastmcp` package, not just the `mcp.server.fastmcp` subset bundled in the
official SDK) is the practical way to build a server: a `FastMCP("name")` instance, and `@mcp.tool` turns
a plain Python function into a protocol-compliant tool — schema generation, validation wiring, and
JSON-RPC framing are handled for you.

```python
from fastmcp import FastMCP
mcp = FastMCP("resumegap_mcp")

@mcp.tool
def my_tool(x: int) -> int:
    return x + 1
```

### 2.3 Pydantic input validation — and *when* it runs
Instead of loose function parameters, real MCP tools validate input with a Pydantic model so the LLM
client gets a proper JSON Schema (types, constraints, descriptions) and bad calls are rejected *before*
your business logic runs:

```python
class ResumeGapInput(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    resume_text: str = Field(..., min_length=20, description="...")
    jd_text: str = Field(..., min_length=20, description="...")
    top_n: int = Field(default=8, ge=1, le=20, description="...")
```

`str_strip_whitespace=True` is not cosmetic — it changes *what reaches your min_length check*. Pydantic
strips whitespace **before** running length validation, so a 25-character string of pure spaces fails
`min_length=20` (post-strip length is 0), not because your code checked for it, but because the model
config already did. Verify this before writing your own redundant blank-check — it may already be dead code.

### 2.4 Single-model-parameter tools are NOT auto-flattened
A tool defined as `def resumegap_analyze(params: ResumeGapInput) -> dict` does **not** expose
`resume_text` / `jd_text` / `top_n` as top-level call arguments. Verified live against `fastmcp==3.4.4`:
`list_tools()` shows the generated `inputSchema` nests every field one level down, under a `"params"` key,
and `call_tool()` must be called as `{"params": {"resume_text": ..., ...}}` — calling with the fields at
the top level raises `missing_argument` / `unexpected_keyword_argument` errors. This is a real,
version-specific behavior to verify with a throwaway script, not something to assume from documentation
examples that show flattened calls.

### 2.5 Testing without a live client: in-memory `Client`
`fastmcp.Client` accepts a `FastMCP` server instance directly (no subprocess, no stdio, no network) —
ideal for a notebook:

```python
from fastmcp import Client
async with Client(mcp) as client:
    tools = await client.list_tools()
    result = await client.call_tool("tool_name", {"params": {...}})
    result.data          # parsed Python object
    result.content        # raw MCP TextContent blocks (JSON string)
```

**Verified gotcha:** wrapping that in `asyncio.run(main())` inside a Jupyter/Colab cell raises
`RuntimeError: asyncio.run() cannot be called from a running event loop` — the kernel already runs one.
Jupyter supports top-level `await`, so call `await main()` directly in the cell instead. `asyncio.run()`
is correct in a plain `.py` script's `if __name__ == "__main__":` block, but not inside a notebook cell.

### 2.6 Errors: `ToolError` vs a raw exception
A Pydantic validation failure (missing field, `min_length`, `le`) is caught automatically and returned to
the client as a `ToolError` with a structured, readable message — never a raw Python traceback. For
failures *inside* your function body that Pydantic can't catch (e.g. a value that's technically valid
input but meaningless, like a resume in the wrong language), raise `fastmcp.exceptions.ToolError`
yourself with an actionable message. Never let an unguarded `AttributeError` or similar leak to the
caller — an LLM client can't act on `'NoneType' object has no attribute 'lower'`.

### 2.7 Tool annotations
`readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint` are metadata hints (not enforced by
the server) that tell a client what kind of tool this is, so it can decide things like whether to ask the
user for confirmation before calling it. A pure analysis tool like ours is read-only, non-destructive,
idempotent, and closed-world (no external API calls).



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 3 &mdash; Practice Guide</h2>
</div>

Build `resumegap_mcp_server.py` as a **separate `.py` file** (not notebook-only code), per project
convention for deployable artifacts. Write it with `%%writefile` in your own cell below (like Raw Data
did), then import and exercise it from later cells.



**Task 1 — `ResumeGapInput` Pydantic model (20 pts)**
Define the input model: `resume_text` (str, required, `min_length=20`), `jd_text` (str, required,
`min_length=20`), `top_n` (int, default `8`, `ge=1`, `le=20`). Use `model_config = ConfigDict(
str_strip_whitespace=True, extra="forbid")`. Give every field a real one-sentence `description=` — this
is what an LLM client actually reads to know how to call your tool.


In [7]:
# ── TASK 1 (20 pts): Define the Pydantic input model ─────────────────────
# Goal: Create a validated input model for the MCP tool.
# Fields: resume_text (required, min_length=20), jd_text (required, min_length=20),
#         top_n (default=8, ge=1, le=20).
# Config: str_strip_whitespace=True, extra="forbid".
# Each field gets a real description for the LLM client.

from pydantic import BaseModel, Field, ConfigDict

class ResumeGapInput(BaseModel):
    model_config = ConfigDict(
        str_strip_whitespace=True,
        extra="forbid"
    )

    resume_text: str = Field(
        ...,
        min_length=20,
        description="The full text of the candidate's resume or CV."
    )
    jd_text: str = Field(
        ...,
        min_length=20,
        description="The full text of the job description to compare against."
    )
    top_n: int = Field(
        default=8,
        ge=1,
        le=20,
        description="Number of top gap words to return (between 1 and 20)."
    )


**Task 2 — Register the tool with FastMCP (25 pts)**
Create `mcp = FastMCP("resumegap_mcp")`. Register a tool named `"resumegap_analyze"` (explicit `name=`,
don't rely on the function name) taking a single `params: ResumeGapInput` argument and returning the
`analyze_resume_gap()` dict directly. Add the four annotations from Concept Notes 2.7. Write a real
docstring (not a placeholder) — it becomes the tool's `description` field that the client sees.


In [8]:
# ── TASKS 2 & 3 (25+20 pts): Tool registration and boundary error handling ──
# Goal: Expose analyze_resume_gap as an MCP tool with annotations,
#       and wrap engine errors in ToolError for clean client responses.
# Method:
#   - Create FastMCP instance.
#   - Use @mcp.tool with explicit name and annotations dict (camelCase keys).
#   - Inside the tool, call the engine; catch any exception and re-raise as ToolError.
#   - Note: Pydantic already validates min_length, bounds, and strips whitespace,
#     so no additional manual guards are needed.

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from resumegap_engine import analyze_resume_gap

mcp = FastMCP("resumegap_mcp")

@mcp.tool(
    name="resumegap_analyze",
    annotations={
        "readOnlyHint": True,       # does not modify state
        "destructiveHint": False,   # does not delete or alter data
        "idempotentHint": True,     # calling repeatedly has same effect
        "openWorldHint": False      # no external side effects
    }
)
def resumegap_analyze(params: ResumeGapInput) -> dict:
    """
    Analyze gaps between a resume and a job description using TF‑IDF similarity.

    Returns a similarity score (0-100), a match band (Strong/Moderate/Weak),
    and a list of top gap words that appear in the JD but not in the resume.
    """
    try:
        return analyze_resume_gap(
            resume=params.resume_text,
            jd=params.jd_text,
            top_n=params.top_n
        )
    except Exception as e:
        # Any unexpected error from the engine is wrapped as a ToolError
        raise ToolError(f"Engine error during analysis: {str(e)}")


**Task 3 — Error handling at the boundary (20 pts)**
Import `ToolError` from `fastmcp.exceptions`. Inside `resumegap_analyze`, before calling the engine,
guard against inputs that are *technically valid to Pydantic but meaningless* — decide for yourself
whether a blank-after-strip check is still reachable given `str_strip_whitespace=True` (Concept Notes
2.3), and justify your answer with a one-line comment rather than just copying a guard clause.


In [9]:
# ── TASKS 2 & 3 (25+20 pts): Tool registration and boundary error handling ──
# Goal: Expose analyze_resume_gap as an MCP tool with annotations,
#       and wrap engine errors in ToolError for clean client responses.
# Method:
#   - Create FastMCP instance.
#   - Use @mcp.tool with explicit name and annotations dict (camelCase keys).
#   - Inside the tool, call the engine; catch any exception and re-raise as ToolError.
#   - Note: Pydantic already validates min_length, bounds, and strips whitespace,
#     so no additional manual guards are needed.

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from resumegap_engine import analyze_resume_gap

mcp = FastMCP("resumegap_mcp")

@mcp.tool(
    name="resumegap_analyze",
    annotations={
        "readOnlyHint": True,       # does not modify state
        "destructiveHint": False,   # does not delete or alter data
        "idempotentHint": True,     # calling repeatedly has same effect
        "openWorldHint": False      # no external side effects
    }
)
def resumegap_analyze(params: ResumeGapInput) -> dict:
    """
    Analyze gaps between a resume and a job description using TF‑IDF similarity.

    Returns a similarity score (0-100), a match band (Strong/Moderate/Weak),
    and a list of top gap words that appear in the JD but not in the resume.
    """
    try:
        return analyze_resume_gap(
            resume=params.resume_text,
            jd=params.jd_text,
            top_n=params.top_n
        )
    except Exception as e:
        # Any unexpected error from the engine is wrapped as a ToolError
        raise ToolError(f"Engine error during analysis: {str(e)}")


**Task 4 — Verify the call shape empirically (20 pts)**
Don't assume how to call your own tool. Use an in-memory `Client` to run `list_tools()`, print the
`inputSchema` as JSON, and from that schema alone (not from Concept Notes) determine the correct shape
for `call_tool()`. Then call `resumegap_analyze` on **all three** pairs in `pairs` with `top_n=5`, and
print each result.


In [10]:
# ── TASK 4 (20 pts): Verify the call shape empirically ──────────────────
# Goal: Use an in‑memory Client to list tools, inspect inputSchema,
#       and call the tool correctly with nested "params".
# Method:
#   - Print the inputSchema to determine the exact required shape.
#   - Call the tool on all three pairs with top_n=5, print results.
#   - Use `await main()` directly (Jupyter already runs an event loop).

import json
from fastmcp import Client

async def main():
    async with Client(mcp) as client:
        # List tools and print the schema for the tool
        tools = await client.list_tools()
        for tool in tools:
            if tool.name == "resumegap_analyze":
                print("Tool inputSchema:")
                print(json.dumps(tool.inputSchema, indent=2))

        # Now call the tool on all three pairs with top_n=5
        for pair_name, pair_data in pairs.items():
            result = await client.call_tool(
                "resumegap_analyze",
                {
                    "params": {
                        "resume_text": pair_data["resume"],
                        "jd_text": pair_data["jd"],
                        "top_n": 5
                    }
                }
            )
            # result.data contains the parsed return value (dict)
            print(f"\n{pair_name} result:")
            print(result.data)

# In a notebook cell, top‑level await is allowed:
await main()

Tool inputSchema:
{
  "additionalProperties": false,
  "properties": {
    "params": {
      "additionalProperties": false,
      "properties": {
        "resume_text": {
          "description": "The full text of the candidate's resume or CV.",
          "minLength": 20,
          "type": "string"
        },
        "jd_text": {
          "description": "The full text of the job description to compare against.",
          "minLength": 20,
          "type": "string"
        },
        "top_n": {
          "default": 8,
          "description": "Number of top gap words to return (between 1 and 20).",
          "maximum": 20,
          "minimum": 1,
          "type": "integer"
        }
      },
      "required": [
        "resume_text",
        "jd_text"
      ],
      "type": "object"
    }
  },
  "required": [
    "params"
  ],
  "type": "object"
}

P1_DataAnalyst result:
{'similarity_score': 17.81, 'match_band': 'Weak Match', 'gap_words': ['aws', 'based', 'clearly', 'cloud', 'communi


**Task 5 — Prove the wrapper is faithful, and test the error paths (15 pts)**
(a) Call `analyze_resume_gap()` directly (no MCP) on `P1_DataAnalyst` with `top_n=5`, and assert the
result is identical to the MCP tool's result for the same pair — the protocol layer must not change the
answer. (b) Trigger and print three distinct rejected calls: `resume_text` under 20 chars, `top_n=50`,
and a missing required field. For each, print the exception type and confirm the message is a structured
validation error, not a raw traceback.


In [11]:
# ── TASK 5 (15 pts): Faithfulness and error‑path tests ──────────────────
# Goal: (a) prove MCP wrapper does not change the answer,
#       (b) trigger and print three distinct validation failures.
# Method:
#   (a) Compare direct engine call with MCP result, assert equality.
#   (b) Use try/except with client.call_tool for three cases:
#       1. resume_text too short (under 20 chars after strip)
#       2. top_n > 20 (le=20 violation)
#       3. missing required field (jd_text omitted)
#   Print exception type and message – should be structured ToolError.

from fastmcp import Client
from resumegap_engine import analyze_resume_gap

async def task5():
    # ---- (a) Faithfulness test ----
    pair = pairs["P1_DataAnalyst"]
    direct = analyze_resume_gap(pair["resume"], pair["jd"], top_n=5)

    async with Client(mcp) as client:
        mcp_result = await client.call_tool(
            "resumegap_analyze",
            {
                "params": {
                    "resume_text": pair["resume"],
                    "jd_text": pair["jd"],
                    "top_n": 5
                }
            }
        )
    assert direct == mcp_result.data, "MCP wrapper changed the answer!"
    print("✅ Faithfulness test passed: direct and MCP results match.")

    # ---- (b) Error path tests ----
    async with Client(mcp) as client:
        # 1. resume_text too short (after strip, length < 20)
        try:
            await client.call_tool(
                "resumegap_analyze",
                {
                    "params": {
                        "resume_text": "too short",
                        "jd_text": "valid but long enough " * 5,   # length > 20
                        "top_n": 5
                    }
                }
            )
        except Exception as e:
            print(f"1. Too-short resume -> {type(e).__name__}: {e}")

        # 2. top_n > 20
        try:
            await client.call_tool(
                "resumegap_analyze",
                {
                    "params": {
                        "resume_text": pair["resume"],
                        "jd_text": pair["jd"],
                        "top_n": 50
                    }
                }
            )
        except Exception as e:
            print(f"2. top_n=50 -> {type(e).__name__}: {e}")

        # 3. Missing required field (jd_text omitted)
        try:
            await client.call_tool(
                "resumegap_analyze",
                {
                    "params": {
                        "resume_text": pair["resume"],
                        "top_n": 5
                    }
                }
            )
        except Exception as e:
            print(f"3. Missing jd_text -> {type(e).__name__}: {e}")

# In a notebook cell, top‑level await is allowed:
await task5()

✅ Faithfulness test passed: direct and MCP results match.


[07/10/26 14:00:10] WARNING  Invalid arguments for tool 'resumegap_analyze': [{'type':               ]8;id=969592;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=714597;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py#1325\1325]8;;\
                             'string_too_short', 'loc': ('params', 'resume_text'), 'msg': 'String                  
                             should have at least 20 characters', 'input': 'too short', 'ctx':                     
                             {'min_length': 20}}]                                                                  

1. Too-short resume -> ToolError: 1 validation error for call[resumegap_analyze]
params.resume_text
  String should have at least 20 characters [type=string_too_short, input_value='too short', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


                    WARNING  Invalid arguments for tool 'resumegap_analyze': [{'type':               ]8;id=511895;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=921149;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py#1325\1325]8;;\
                             'less_than_equal', 'loc': ('params', 'top_n'), 'msg': 'Input should be                
                             less than or equal to 20', 'input': 50, 'ctx': {'le': 20}}]                           

2. top_n=50 -> ToolError: 1 validation error for call[resumegap_analyze]
params.top_n
  Input should be less than or equal to 20 [type=less_than_equal, input_value=50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


                    WARNING  Invalid arguments for tool 'resumegap_analyze': [{'type': 'missing',    ]8;id=725832;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=583600;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/server.py#1325\1325]8;;\
                             'loc': ('params', 'jd_text'), 'msg': 'Field required', 'input':                       
                             {'resume_text': 'Data Analyst with 3 years experience in SQL, Excel,                  
                             and Power BI dashboards. Built weekly sales reports and cleaned raw                   
                             data using pandas. Comfortable presenting insights to stakeholders.',                 
                             'top_n': 5}}]                                                                         

3. Missing jd_text -> ToolError: 1 validation error for call[resumegap_analyze]
params.jd_text
  Field required [type=missing, input_value={'resume_text': 'Data Ana...keholders.', 'top_n': 5}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing



**Interview framing**
*"Walk me through what happens, step by step, when an LLM agent calls your `resumegap_analyze` tool with
a resume that's just three words long."* Draft your own 2-4 sentence answer before checking the Answer
Key discussion below. A strong answer names the exact validation layer that catches it (Pydantic
`min_length`, not your own code) and what the client actually receives back.



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 4 &mdash; Scoring Rubric</h2>
</div>

| Task | Points | Criteria |
|---|---|---|
| 1. `ResumeGapInput` model | 20 | Correct field constraints, `model_config`, real descriptions (not placeholders) |
| 2. Tool registration | 25 | Explicit `name=`, all 4 annotations correct, real docstring, returns engine dict unchanged |
| 3. Error handling | 20 | `ToolError` imported and used correctly; reachability of any manual guard is justified, not assumed |
| 4. Empirical call-shape verification | 20 | Schema printed and read *before* writing the call; correct nested `"params"` calls on all 3 pairs, outputs match |
| 5. Faithfulness + error-path tests | 15 | Assertion passes; all 3 error cases produce `ToolError` (not raw exceptions), messages printed and correct |
| **Total** | **100** | |

**Key Takeaway:** An MCP tool's real interface is whatever the generated `inputSchema` says it is — not
whatever your function signature looks like, and not whatever a tutorial's example shows. A single
Pydantic-model parameter nests every field under that parameter's name in both the schema and the call,
and `str_strip_whitespace` plus `min_length` together can make a defensive check you wrote yourself
completely unreachable. Verify the actual schema and actual error path with a throwaway script before
trusting either — the same discipline that already caught the `interrupt()` and same-step-staleness bugs
in Week 1 applies just as directly to a new library.
